# 01 — Limpieza y carga a SQLite

Toma el CSV crudo de Kaggle, lo limpia y lo carga en una base de datos SQLite lista para consultar.

**Input:** `data/raw/steam.csv`  
**Output:** `data/clean/steam.db`

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

In [ ]:
df = pd.read_csv('../data/raw/steam.csv', low_memory=False)
print(f'Shape: {df.shape}')
df.head(3)

## Exploración inicial

In [ ]:
nulos = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print('% nulos por columna:')
print(nulos[nulos > 0].round(1))

In [ ]:
print(df.dtypes)
df.describe()

## Selección de columnas

De las columnas disponibles, seleccionamos las relevantes para el análisis de valor (precio, playtime, reviews, género).

In [ ]:
COLS = [
    'appid', 'name', 'developer', 'publisher',
    'genres', 'release_date',
    'price', 'positive_ratings', 'negative_ratings',
    'average_playtime', 'median_playtime',
    'owners'
]

cols_ok = [c for c in COLS if c in df.columns]
df = df[cols_ok].copy()
print(f'Columnas seleccionadas: {cols_ok}')

## Corrección de tipos

In [ ]:
numeric_cols = ['price', 'positive_ratings', 'negative_ratings',
                'average_playtime', 'median_playtime']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# release_date → año como entero
if 'release_date' in df.columns:
    df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year.astype('Int64')

print(df.dtypes)

## Limpieza de nulos y filas inválidas

In [ ]:
filas_orig = len(df)

# Sin nombre o género no se puede analizar
df = df.dropna(subset=['name', 'genres'])

# Precio nulo → asumir gratuito (0.0)
df['price'] = df['price'].fillna(0.0)

# Playtime y ratings nulos → 0
for col in ['average_playtime', 'median_playtime', 'positive_ratings', 'negative_ratings']:
    if col in df.columns:
        df[col] = df[col].fillna(0)

print(f'Filas eliminadas: {filas_orig - len(df):,}')
print(f'Filas restantes:  {len(df):,}')

## Columnas derivadas

Creamos métricas que no existen en el dataset original y que son el corazón del análisis.

In [ ]:
total_ratings = df['positive_ratings'] + df['negative_ratings']

# Review score: % positivo (solo juegos con al menos 10 reviews)
df['review_score'] = np.where(
    total_ratings >= 10,
    (df['positive_ratings'] / total_ratings * 100).round(1),
    np.nan
)

df['total_ratings'] = total_ratings

# Horas por dólar: métrica de valor central del proyecto
# Solo para juegos de pago; evitar división por cero
df['horas_por_dolar'] = np.where(
    df['price'] > 0,
    (df['average_playtime'] / df['price']).round(1),
    np.nan
)

# Categoría de precio
df['categoria_precio'] = pd.cut(
    df['price'],
    bins=[-0.1, 0, 5, 15, 30, 60, float('inf')],
    labels=['gratuito', 'menos de $5', '$5–$15', '$15–$30', '$30–$60', 'más de $60']
)

# Métrica compuesta de valor (0–100)
# Combina review score, horas por dólar y popularidad (ratings).
# Solo se calcula para juegos de pago con suficientes reviews.
mask = (df['price'] > 0) & (df['review_score'].notna()) & (total_ratings >= 10)

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

df_val = df[mask][['review_score', 'horas_por_dolar', 'total_ratings']].copy()
df_val['horas_por_dolar'] = df_val['horas_por_dolar'].clip(0, 500)
df_val['total_ratings']   = df_val['total_ratings'].clip(0, 50000)

scaled = scaler.fit_transform(df_val)
# Pesos: review 50%, horas/dólar 35%, popularidad 15%
valor_score = scaled @ np.array([0.50, 0.35, 0.15]) * 100

df['valor_score'] = np.nan
df.loc[mask, 'valor_score'] = valor_score.round(1)

print('Columnas derivadas creadas: review_score, total_ratings, horas_por_dolar, categoria_precio, valor_score')
df[['name', 'price', 'review_score', 'horas_por_dolar', 'valor_score']].head(5)

## Tratamiento de outliers en precio y playtime

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_pago = df[df['price'] > 0]
axes[0].hist(df_pago['price'], bins=60, color='#5b8db8', alpha=0.8, edgecolor='white')
axes[0].set_title('Distribución de precios (juegos de pago)')
axes[0].set_xlabel('Precio (USD)')

axes[1].hist(df['average_playtime'].clip(0, 5000), bins=60, color='#4a9e6b', alpha=0.8, edgecolor='white')
axes[1].set_title('Distribución de playtime (clip a 5000h)')
axes[1].set_xlabel('Horas promedio')

plt.tight_layout()
plt.savefig('../img/01_distribuciones.png', dpi=150)
plt.show()

print(f"Juegos gratuitos: {(df['price'] == 0).sum():,}")
print(f"Juegos de pago:   {(df['price'] > 0).sum():,}")
print(f"Precio máximo:    ${df['price'].max():,.2f}")

## Carga a SQLite

In [ ]:
DB_PATH = '../data/clean/steam.db'

conn = sqlite3.connect(DB_PATH)
df.to_sql('games', conn, if_exists='replace', index=False)

# Índices para acelerar las queries del análisis
conn.execute('CREATE INDEX IF NOT EXISTS idx_genres ON games(genres)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_price ON games(price)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_year ON games(release_year)')
conn.commit()
conn.close()

print(f'Base de datos creada: {DB_PATH}')
print(f'Tabla: games — {len(df):,} filas × {df.shape[1]} columnas')

## Resumen de la limpieza

| Decisión | Columna | Criterio |
|---|---|---|
| Eliminar filas | `name`, `genres` | Sin estos campos no se puede analizar el juego |
| fillna(0) | `price` | Nulo = gratuito en Steam |
| fillna(0) | `playtime`, `ratings` | Nulo = sin datos registrados |
| Métrica nueva | `review_score` | % positivo, solo si tiene ≥10 reviews |
| Métrica nueva | `horas_por_dolar` | Playtime / precio — métrica central del proyecto |
| Métrica nueva | `categoria_precio` | Segmentación en 6 rangos de precio |
| Métrica nueva | `valor_score` | Score compuesto 0–100: review 50% + horas/$ 35% + popularidad 15% |